# درس ۰۸ - الگوی طراحی چندعامل


## تنظیمات


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## چرا سیستم‌های چندعاملی؟

کارهای دنیای واقعی مانند برنامه‌ریزی سفر شامل انواع مختلفی از تخصص‌ها هستند — لجستیک، دانش محلی، بودجه‌بندی و غیره. یک عامل واحد که سعی در انجام همه این‌ها داشته باشد به سرعت غیرقابل مدیریت می‌شود.

سیستم‌های چندعاملی این مشکل را از طریق **تخصص** حل می‌کنند: هر عامل روی یک حوزه تخصصی تمرکز می‌کند و نتایج با کیفیت‌تری نسبت به یک فرد عمومی تولید می‌کند. آنها همچنین **قابلیت مقیاس‌پذیری** را بهبود می‌بخشند — می‌توانید عوامل جدیدی اضافه کنید (مثلاً یک متخصص پرواز، یک منتقد رستوران) بدون نیاز به بازنویسی جریان کاری موجود. عوامل از طریق یک خط لوله ساختاریافته با هم ترکیب می‌شوند و زمینه را از یکی به دیگری منتقل می‌کنند.


## ایجاد عوامل تخصصی


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ساخت یک جریان کاری ترتیبی

`WorkflowBuilder` به شما امکان می‌دهد عوامل را به یک گراف جهت‌دار متصل کنید. در اینجا ما یک خط لوله ساده دو مرحله‌ای ایجاد می‌کنیم: **TravelPlanner** برنامه سفر را طرح‌ریزی می‌کند، سپس **TravelConcierge** آن را مرور و بهبود می‌بخشد.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## افزودن نمایندگان بیشتر به جریان کاری

یکی از بزرگ‌ترین مزایای الگوی چندنماینده‌ای، سهولت گسترش آن است. در ادامه یک نماینده **BudgetReviewer** اضافه می‌کنیم که طرح را با بودجه مسافر بررسی می‌کند، مواردی که ممکن است هزینه‌ها را فراتر از حد مجاز ببرد علامت‌گذاری می‌کند و جایگزین‌های صرفه‌جویی در هزینه پیشنهاد می‌دهد. جریان کاری اکنون سه نماینده را به ترتیب اجرا می‌کند:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## خلاصه

در این درس یاد گرفتید چگونه:

1. **عامل‌های تخصصی ایجاد کنید** — هر کدام با نقشی متمرکز (برنامه‌ریزی، پذیرش، بررسی بودجه).
2. **عامل‌ها را به یک جریان کاری متوالی متصل کنید** با استفاده از `WorkflowBuilder` و `add_edge`.
3. **خروجی را از یک خط لوله چندعاملی به‌صورت جریانی منتقل کنید** و دنبال کنید که کدام عامل در حال صحبت است.
4. **یک جریان کاری را گسترش دهید** با افزودن عامل‌های جدید به زنجیره بدون اینکه عامل‌های موجود تغییر کنند.

الگوی طراحی چندعاملی هر عامل را ساده نگه می‌دارد در حالی که نتایجی غنی‌تر و دقیق‌تر ارائه می‌دهد که هیچ عامل تکی نمی‌تواند به تنهایی به آن دست یابد.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:  
این سند با استفاده از سرویس ترجمه ماشینی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی اشتباهات یا نواقصی باشند. سند اصلی به زبان بومی خود، منبع معتبر و رسمی تلقی می‌شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما مسئول هرگونه سوءتفاهم یا تفسیر نادرست ناشی از استفاده از این ترجمه نیستیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
